In [ ]:
!pip install opencv-python roboflow imageio imageio-ffmpeg -q

In [ ]:
from google.colab import drive
import roboflow
import cv2
import os
import time
from pathlib import Path
from tqdm import tqdm
import imageio

drive.mount('/content/drive')

In [ ]:
def get_extracted_filenames(output_dir: str) -> set:
    if not os.path.exists(output_dir):
        return set()

    existing = {
        f for f in os.listdir(output_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(existing)} file sudah ada di output_dir")
    return existing

In [ ]:
def get_frames_to_extract(total_frames: int, every_n_frames: int, fmt: str) -> dict:
    frames = {}
    extracted_count = 0
    for frame_count in range(total_frames):
        if frame_count % every_n_frames == 0:
            fname = f"frame_{extracted_count:06d}.{fmt}"
            frames[fname] = frame_count
            extracted_count += 1
    return frames

In [ ]:
def get_frames_to_skip(all_frames: dict, existing_files: set) -> dict:
    to_extract = {
        fname: idx
        for fname, idx in all_frames.items()
        if fname not in existing_files
    }
    already_exists = len(all_frames) - len(to_extract)

    print(f"\n  Ringkasan:")
    print(f"  Sudah diekstrak : {already_exists} frame")
    print(f"  Belum diekstrak : {len(to_extract)} frame")
    return to_extract

In [ ]:
def extract_frames_opencv(video_path: str, output_dir: str, to_extract: dict,
                          fmt: str = 'jpg', quality: int = 95) -> bool:
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return False

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\n📹 Informasi Video:")
    print(f"   Total Frames : {total_frames}")
    print(f"   FPS          : {fps}")
    print(f"   Resolusi     : {width}x{height}")
    print(f"   Durasi       : {total_frames/fps:.2f} detik")

    target_indices = {idx: fname for fname, idx in to_extract.items()}

    extracted, skipped = 0, 0
    frame_count = 0

    with tqdm(total=total_frames, desc="🎬 Extracting") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count in target_indices:
                fname    = target_indices[frame_count]
                filepath = os.path.join(output_dir, fname)

                if fmt.lower() == 'png':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_PNG_COMPRESSION, 9])
                elif fmt.lower() == 'jpg':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_JPEG_QUALITY, quality])

                extracted += 1
            else:
                skipped += 1

            frame_count += 1
            pbar.update(1)

    cap.release()

    print(f"\n✅ Selesai!")
    print(f"   Diekstrak : {extracted} frame")
    print(f"   Dilewati  : {skipped} frame (sudah ada / tidak perlu)")
    print(f"   Output    : {output_dir}")
    return True

In [ ]:
def main_extract_frame(video_path: str, output_dir: str, every_n_frames: int,
         fmt: str, quality: int) -> None:

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print("📂 Mengecek file yang sudah ada di output_dir...")
    existing_files = get_extracted_filenames(output_dir)

    print("\n🔢 Menghitung frame yang akan diekstrak...")
    all_frames = get_frames_to_extract(total_frames, every_n_frames, fmt)

    to_extract = get_frames_to_skip(all_frames, existing_files)

    if not to_extract:
        print("\n✅ Semua frame sudah diekstrak, tidak ada yang perlu diproses.")
        return

    extract_frames_opencv(video_path, output_dir, to_extract, fmt, quality)

In [ ]:
def _search_batch_page(project, batch_id: str, offset: int, per_page: int,
                        fields: list, max_retries: int = 5, base_delay: float = 2.0):
    """
    Panggil project.search() untuk satu halaman dengan retry + exponential backoff.
    Ini PENTING karena jika satu request gagal (timeout/rate-limit) di tengah
    pagination ribuan foto, kita tidak boleh langsung berhenti (break) karena
    itu akan membuat daftar nama file yang ter-fetch menjadi TIDAK LENGKAP,
    yang berakibat file yang sebenarnya sudah ada di Roboflow dianggap
    'belum diupload' lalu diupload ulang.

    Return:
        list hasil, atau None jika semua percobaan gagal.
    """
    for attempt in range(1, max_retries + 1):
        try:
            results = project.search(
                batch=True,
                batch_id=batch_id,
                offset=offset,
                limit=per_page,
                fields=fields,
            )
            if isinstance(results, dict):
                results = results.get("results", [])
            return results
        except Exception as e:
            if attempt == max_retries:
                print(f"  [ERROR] Gagal di offset {offset} setelah {max_retries}x percobaan: {e}")
                return None
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  [WARN] Gagal di offset {offset} (percobaan {attempt}/{max_retries}): {e}. "
                  f"Retry dalam {delay:.1f}s...")
            time.sleep(delay)
    return None

In [ ]:
def fetch_all_batch_names(project, batch_id: str, total_images: int,
                           per_page: int = 100) -> tuple[set, bool]:
    """
    Ambil SEMUA nama file dalam sebuah batch Roboflow lewat pagination offset/limit.

    Fungsi ini mengumpulkan nama-nama file yang sudah ada di batch tersebut ke
    dalam sebuah set. Untuk mendiagnosis apakah pagination offset/limit
    benar-benar overlap antar-halaman (yaitu nama file yang sama muncul lagi
    di halaman/offset berikutnya — ini adalah root cause dugaan sebelumnya
    untuk 'duplikat palsu'), setiap halaman diperiksa: berapa banyak nama di
    halaman ini yang SUDAH PERNAH muncul di halaman-halaman SEBELUMNYA
    (overlap_with_previous). Jika angka ini > 0 secara konsisten, itu
    konfirmasi kuat bahwa API mengembalikan item yang sama berulang kali di
    offset berbeda (root cause dari 'terfetch 9334 tapi nama unik cuma 7262'
    padahal batch aslinya tidak punya duplikat nama).

    Ini murni untuk DIAGNOSA/LOGGING — tidak ada penghapusan atau perubahan
    keputusan upload berdasarkan overlap ini. existing_filenames yang
    dikembalikan tetap union dari seluruh nama yang pernah terlihat (dedup
    otomatis lewat set), yang tetap valid untuk existence-check.

    ⚠️ Catatan teknis pagination:
    project.search() TIDAK menjamin akan mengembalikan hasil sebanyak `limit`
    yang diminta (API bisa mem-cap ke angka lebih kecil). Karena itu offset
    dimajukan sebesar JUMLAH HASIL YANG BENAR-BENAR DITERIMA (len(results)),
    bukan sebesar per_page yang diminta, dan loop berhenti hanya ketika hasil
    kosong atau total ter-fetch sudah mencapai total_images.

    Return:
        (names, complete) — names: set semua nama file unik yang ter-fetch.
        complete: True jika seluruh pagination selesai tanpa error fatal.
    """
    names = set()
    fetched_count = 0
    offset = 0
    total_overlap = 0

    while True:
        results = _search_batch_page(project, batch_id, offset, per_page, fields=["name"])

        if results is None:
            print(f"  ⚠️  PERINGATAN: Gagal mengambil data di offset {offset} setelah retry. "
                  f"Data KEMUNGKINAN TIDAK LENGKAP ({fetched_count}/{total_images} terfetch).")
            return names, False

        if not results:
            break

        page_names = [os.path.basename(item.get("name") or item.get("filename") or "") for item in results]

        # ── Diagnosa overlap: berapa nama di halaman ini yang sudah pernah
        # muncul di halaman-halaman SEBELUMNYA (bukan duplikat di dalam
        # halaman yang sama, tapi overlap ANTAR offset/pagination). ──
        page_overlap_names = [n for n in page_names if n in names]
        overlap_in_page = len(page_overlap_names)
        total_overlap += overlap_in_page

        names.update(page_names)

        fetched_count += len(results)

        overlap_flag = f" ⚠️ OVERLAP: {overlap_in_page} nama sudah muncul di offset sebelumnya!" if overlap_in_page > 0 else ""
        print(f"  [Offset {offset:>5}] → terfetch: {fetched_count}/{total_images} "
              f"(nama unik kumulatif: {len(names)}){overlap_flag}")

        if overlap_in_page > 0:
            contoh = page_overlap_names[:5]
            print(f"      Contoh nama yang overlap: {contoh}"
                  f"{' ...' if overlap_in_page > 5 else ''}")

        offset += len(results)

        if total_images and fetched_count >= total_images:
            break

    if total_overlap > 0:
        print(f"\n  🔎 DIAGNOSA: Total {total_overlap} kemunculan nama yang overlap antar-offset "
              f"terdeteksi dari {fetched_count} item yang terfetch, tapi hanya {len(names)} nama unik.\n"
              f"  Ini mengonfirmasi bahwa pagination offset/limit pada project.search() "
              f"mengembalikan item yang SAMA berulang kali di offset yang berbeda "
              f"(root cause dari selisih 'terfetch' vs 'nama unik'), BUKAN karena "
              f"batch di Roboflow benar-benar memiliki duplikat nama file.")
    else:
        print("\n  🔎 DIAGNOSA: Tidak ada overlap nama antar-offset terdeteksi pada run ini.")

    return names, True

In [ ]:
def get_roboflow_filenames(project, batch_name: str) -> set:
    """
    Ambil semua nama file unik yang ada di sebuah batch Roboflow.

    Return:
        set nama file yang sudah ada di batch tersebut.
    """
    batch_id = None
    total_images = 0

    batches = project.get_batches().get("batches", [])
    for batch in batches:
        if batch.get("name") == batch_name:
            batch_id = batch.get("id")
            total_images = batch.get("images", 0)
            break

    if not batch_id:
        print(f"  [ERROR] Batch '{batch_name}' tidak ditemukan.")
        return set()

    print(f"  → Batch ditemukan: id={batch_id}, total={total_images} foto")

    names, complete = fetch_all_batch_names(project, batch_id, total_images)

    if not complete:
        print("  ⚠️  Data nama file di batch TIDAK LENGKAP karena pagination gagal di tengah jalan. "
              "Sebaiknya jalankan ulang fungsi ini sebelum melanjutkan ke upload.")

    print(f"\n  📊 Nama file unik di batch Roboflow: {len(names)}")
    return names

In [ ]:
def get_local_filenames(output_dir: str) -> dict:
    local_files = {
        filename: os.path.join(output_dir, filename)
        for filename in os.listdir(output_dir)
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(local_files)} file ditemukan di lokal")
    return local_files

In [ ]:
def get_files_to_upload(local_files: dict, existing_filenames: set) -> dict:
    """
    Kembalikan file lokal yang namanya TIDAK ADA di batch Roboflow.

    Logika sengaja dibuat sesederhana mungkin: hanya set difference antara
    nama file di Drive dan nama file yang sudah ada di batch. Tidak ada
    penghitungan/penanganan duplikat sama sekali di sini.
    """
    to_upload = {
        filename: path
        for filename, path in local_files.items()
        if filename not in existing_filenames
    }

    print(f"\n  📊 Ringkasan:")
    print(f"     File di Drive           : {len(local_files)}")
    print(f"     Nama file di batch RF    : {len(existing_filenames)}")
    print(f"     Belum ada di batch (upload): {len(to_upload)}")
    return to_upload

In [ ]:
def upload_files_to_roboflow(project, to_upload: dict, batch_name: str) -> None:
    uploaded, failed = 0, 0

    for filename, image_path in to_upload.items():
        print(f"  [UPLOAD] {filename}")
        try:
            project.upload(
                image_path=image_path,
                batch_name=batch_name,
                num_retry_uploads=3
            )
            uploaded += 1
        except Exception as e:
            print(f"  [ERROR]  {filename} → {e}")
            failed += 1

    print(f"\nSelesai → Berhasil: {uploaded} | Gagal: {failed} | Dilewati: {len(to_upload) - uploaded}")

In [ ]:
def main_upload_to_roboflow(project, output_dir: str, batch_name: str) -> None:
    existing_filenames = get_roboflow_filenames(project, batch_name)
    local_files = get_local_filenames(output_dir)

    to_upload = get_files_to_upload(local_files, existing_filenames)

    if not to_upload:
        print("\n✅ Semua file sudah ada di Roboflow, tidak ada yang perlu diupload.")
        return

    print("\nMemulai upload...")
    upload_files_to_roboflow(project, to_upload, batch_name)

In [ ]:
rf = roboflow.Roboflow(api_key="hRSsLIOXzzSZPkM0Wd54")
project = rf.workspace().project("plastic-trash-detection-p9gqv")
filename = "0412"
MAIN_PATH = "/content/drive/MyDrive/"
# VIDEO_PATH = f"{MAIN_PATH}/DJI_{filename}.MOV" # bisa .MOV atau .MP4
OUTPUT_DIR = f"{MAIN_PATH}/DJISampah/{filename}"
EXTRACT_EVERY_N_FRAMES = 1  # 1 = semua frame, 2 = setiap frame ke-2, dst
OUTPUT_FORMAT = 'jpg'        # 'png' (lossless) atau 'jpg' (lossy)
QUALITY = 98                 # Untuk JPG (1-100)

In [ ]:
# main_extract_frame(VIDEO_PATH, OUTPUT_DIR, EXTRACT_EVERY_N_FRAMES, OUTPUT_FORMAT, QUALITY)

In [ ]:
main_upload_to_roboflow(project, OUTPUT_DIR, filename)